In [1]:
import pandas as pd
from Bio import SeqIO, Entrez
from Bio.Seq import Seq 
import matplotlib as mpl
from matplotlib.font_manager import FontProperties
mpl.rcParams['svg.fonttype'] = 'none'
# Path to your Arial.ttf file
# arial_font_path = '/home/skawakubo/fonts/Arial.ttf'
# Create a FontProperties object for the Arial font
# arial_font = FontProperties(fname=arial_font_path)
# mpl.rcParams['font.family'] = arial_font.get_name()
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time
import subprocess

In [2]:
def fasta_to_dataframe(fasta_file):
    ids = []
    sequences = []
    for record in SeqIO.parse(fasta_file, "fasta"):
        ids.append(record.id)
        sequences.append(str(record.seq))

    df = pd.DataFrame({
        "ID": ids,
        "seq": sequences
    })
    return df

In [3]:
# NCBI + Izumi のfasta
seq_path = '/Users/izumikida/FCoV_seq_relatives/FCoV_relatives_Spike/FCoV_relatives_type1_withoutgap_trim.fasta'
seq = fasta_to_dataframe(seq_path)
seq.rename(columns={"ID": "Accession"}, inplace=True)
seq

,Accession,seq
0,PQ416775.1,atgatagtgttaatacttgcacttcttagcaccgccagctctgaag...
1,PX315748.1,atgatagtgctaatacttgcattcgtaagtgtcgttagttctaacg...
2,PX315749.1,atgatagtgctaatacttgcattcgtaagtgttgttagttctcacg...
3,PX315750.1,atgatagtgctaatacttgcattcgtaagtgttgttagttctcacg...
4,PX315751.1,atgatagtgttaatacttgcactccttagctttgctaggtctgacg...
...,...,...
150,MT821905.1,atgaagtttttaat--------------------------------...
151,MN692770.1,atgaaggttttaat--------------------------------...
152,LT545990.1,atgaagtttttaat--------------------------------...
153,KR061459.1,atgaagtttttaat--------------------------------...


In [4]:
# Read the metadata　blastでhitした全配列のmeta
meta_path = '/Users/izumikida/FCoV_seq_relatives/FCoV_relatives_meta_20260223.tsv'
meta = pd.read_csv(meta_path, sep="\t")
meta

,Accession,biotype,Organism_Name,Isolate,Species,Genotype,Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source,virus_type
0,NC_002306.3,FIPV,Feline infectious peritonitis virus,79-1146,Alphacoronavirus suis,2,29355,USA,USA,NaN,NaN,NaN,2000-05-11,NCBI,FCoV
1,PV941793.1,FIPV,Feline coronavirus,NaN,Alphacoronavirus suis,NaN,29352,USA: California,USA,Felis catus,NaN,2002,2026-01-22,NCBI,FCoV
2,PX471051.1,FIPV,Feline coronavirus,AH2023,Alphacoronavirus suis,NaN,29013,China,China,Feliformia,peritoneal effusion,2023-03-15,2025-10-28,NCBI,FCoV
3,PQ416775.1,FIPV,Feline coronavirus,H30,Alphacoronavirus suis,NaN,29201,China: Shandong,China,Felis catus,liver,2023-11-05,2025-10-01,NCBI,FCoV
4,PX315748.1,FIPV,Feline coronavirus,fip_37_liver,Alphacoronavirus suis,1,28497,United Kingdom,United Kingdom,Felis catus,liver,2011-10,2025-09-24,NCBI,FCoV
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399,MT821905.1,SeCoV,Swine enteric coronavirus,NaN,Alphacoronavirus suis,NaN,28081,Italy,Italy,Sus scrofa,NaN,2019-03-04,2022-04-30,NCBI,AlphaCoV1
400,MN692770.1,SeCoV,Swine enteric coronavirus,SeCoV-1480-Murcia-Lorca,Alphacoronavirus suis,NaN,28094,Spain,Spain,Sus scrofa,NaN,2014-05-30,2020-06-24,NCBI,AlphaCoV1
401,LT545990.1,SeCoV,Swine enteric coronavirus,SeCoV/GER/L00930/2012,Alphacoronavirus suis,NaN,28050,Germany,Germany,Sus scrofa,NaN,2012,2016-05-14,NCBI,AlphaCoV1
402,KR061459.1,SeCoV,Swine enteric coronavirus,NaN,Alphacoronavirus suis,NaN,28111,Italy,Italy,Sus scrofa,NaN,2009-09-04,2015-12-13,NCBI,AlphaCoV1


In [5]:
# Merge the metadata and the sequence
FCoV1 = pd.merge(meta, seq, on="Accession")
FCoV1

,Accession,biotype,Organism_Name,Isolate,Species,Genotype,Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source,virus_type,seq
0,PQ416775.1,FIPV,Feline coronavirus,H30,Alphacoronavirus suis,NaN,29201,China: Shandong,China,Felis catus,liver,2023-11-05,2025-10-01,NCBI,FCoV,atgatagtgttaatacttgcacttcttagcaccgccagctctgaag...
1,PX315748.1,FIPV,Feline coronavirus,fip_37_liver,Alphacoronavirus suis,1,28497,United Kingdom,United Kingdom,Felis catus,liver,2011-10,2025-09-24,NCBI,FCoV,atgatagtgctaatacttgcattcgtaagtgtcgttagttctaacg...
2,PX315749.1,FIPV,Feline coronavirus,fip_68_liver,Alphacoronavirus suis,1,28550,United Kingdom,United Kingdom,Felis catus,liver,2013-01-11,2025-09-24,NCBI,FCoV,atgatagtgctaatacttgcattcgtaagtgttgttagttctcacg...
3,PX315750.1,FIPV,Feline coronavirus,fip_68_mln,Alphacoronavirus suis,1,29246,United Kingdom,United Kingdom,Felis catus,lymph,2013-01-11,2025-09-24,NCBI,FCoV,atgatagtgctaatacttgcattcgtaagtgttgttagttctcacg...
4,PX315751.1,FIPV,Feline coronavirus,fip_86_liver,Alphacoronavirus suis,1,29225,United Kingdom,United Kingdom,Felis catus,liver,2013-06-07,2025-09-24,NCBI,FCoV,atgatagtgttaatacttgcactccttagctttgctaggtctgacg...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150,MT821905.1,SeCoV,Swine enteric coronavirus,NaN,Alphacoronavirus suis,NaN,28081,Italy,Italy,Sus scrofa,NaN,2019-03-04,2022-04-30,NCBI,AlphaCoV1,atgaagtttttaat--------------------------------...
151,MN692770.1,SeCoV,Swine enteric coronavirus,SeCoV-1480-Murcia-Lorca,Alphacoronavirus suis,NaN,28094,Spain,Spain,Sus scrofa,NaN,2014-05-30,2020-06-24,NCBI,AlphaCoV1,atgaaggttttaat--------------------------------...
152,LT545990.1,SeCoV,Swine enteric coronavirus,SeCoV/GER/L00930/2012,Alphacoronavirus suis,NaN,28050,Germany,Germany,Sus scrofa,NaN,2012,2016-05-14,NCBI,AlphaCoV1,atgaagtttttaat--------------------------------...
153,KR061459.1,SeCoV,Swine enteric coronavirus,NaN,Alphacoronavirus suis,NaN,28111,Italy,Italy,Sus scrofa,NaN,2009-09-04,2015-12-13,NCBI,AlphaCoV1,atgaagtttttaat--------------------------------...


In [6]:
print(FCoV1["virus_type"].value_counts())

virus_type
FCoV         149
AlphaCoV1      5
CCoV           1
Name: count, dtype: int64


In [7]:
# virus_type が 'FCoV' の行のみを抽出
FCoV1 = FCoV1[FCoV1['virus_type'] == 'FCoV']

# 以前の処理と同様に、biotype の欠損値を除去してインデックスをリセット
FCoV1 = FCoV1.dropna(subset=['biotype'])
FCoV1.reset_index(drop=True, inplace=True)

FCoV1

,Accession,biotype,Organism_Name,Isolate,Species,Genotype,Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source,virus_type,seq
0,PQ416775.1,FIPV,Feline coronavirus,H30,Alphacoronavirus suis,NaN,29201,China: Shandong,China,Felis catus,liver,2023-11-05,2025-10-01,NCBI,FCoV,atgatagtgttaatacttgcacttcttagcaccgccagctctgaag...
1,PX315748.1,FIPV,Feline coronavirus,fip_37_liver,Alphacoronavirus suis,1,28497,United Kingdom,United Kingdom,Felis catus,liver,2011-10,2025-09-24,NCBI,FCoV,atgatagtgctaatacttgcattcgtaagtgtcgttagttctaacg...
2,PX315749.1,FIPV,Feline coronavirus,fip_68_liver,Alphacoronavirus suis,1,28550,United Kingdom,United Kingdom,Felis catus,liver,2013-01-11,2025-09-24,NCBI,FCoV,atgatagtgctaatacttgcattcgtaagtgttgttagttctcacg...
3,PX315750.1,FIPV,Feline coronavirus,fip_68_mln,Alphacoronavirus suis,1,29246,United Kingdom,United Kingdom,Felis catus,lymph,2013-01-11,2025-09-24,NCBI,FCoV,atgatagtgctaatacttgcattcgtaagtgttgttagttctcacg...
4,PX315751.1,FIPV,Feline coronavirus,fip_86_liver,Alphacoronavirus suis,1,29225,United Kingdom,United Kingdom,Felis catus,liver,2013-06-07,2025-09-24,NCBI,FCoV,atgatagtgttaatacttgcactccttagctttgctaggtctgacg...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144,h7,FIPV,Feline coronavirus,h7,Alphacoronavirus suis,1,29065,Hokkaido,Japan,Felis catus,peritoneal effusion,2023-07-04,NaN,Izumi,FCoV,atgatactgttaatatttgcattacttagcgtcgctaagtccgaag...
145,h2,FIPV,Feline coronavirus,h2,Alphacoronavirus suis,1,29273,Hokkaido,Japan,Felis catus,peritoneal effusion,2023-03-24,NaN,Izumi,FCoV,atgatactgttaatatttgcattacttagcgtcgctaagtccgaag...
146,r3,FIPV,Feline coronavirus,r3,Alphacoronavirus suis,1,27502,Hokkaido,Japan,Felis catus,peritoneal effusion,2024-04-24,NaN,Izumi,FCoV,atgatagtgttgatactagtcctctttagcactgttaggtctgaag...
147,v1,FIPV,Feline coronavirus,v1,Alphacoronavirus suis,1,29285,Saitama,Japan,Felis catus,peritoneal effusion,2023-09-07,NaN,Izumi,FCoV,atgatagtgttaatacttgcatttctaagcatcgctaagtctgaag...


In [9]:
# 2. TSV出力
FCoV1.to_csv("FCoV1_full_meta.tsv", sep='\t', index=False)

# 3. FASTA出力
with open("FCoV1_full.fasta", "w") as f:
    for _, row in FCoV1.iterrows():
        f.write(f">{row['Accession']}\n{row['seq']}\n")
print("FASTA file has been saved.")

FASTA file has been saved.
